In [ ]:
import numpy as np
import json
import random

In [ ]:
def calculate_greedy_cost(p_prev, p_curr, p_next, c=0):
    return abs(p_prev - p_curr)
# revise based on previous level

def calculate_planning_cost(p_prev, p_curr, p_next, c=0):

    dist_step_1 = abs(p_prev - p_curr)
    dist_step_2 = abs(p_curr - p_next)
    

    v1 = p_curr - p_prev
    v2 = p_next - p_curr
    
    switch_penalty = c if (v1 * v2 < 0) else 0
    
    return dist_step_1 + dist_step_2 + switch_penalty

In [ ]:
def generate_levels(num_levels=50, screen_width=600, 
                    greedy_func=calculate_greedy_cost, 
                    planning_func=calculate_planning_cost, 
                    c=0,
                    degree_of_conflict=1.5):
    # generate levels wo regard to screen size, scale up based 
    # set doc to 1
    experiment_configs = []
    
    while len(experiment_configs) < num_levels:
        # level 1
        entry = random.randint(100, 500)
        
        # second level
        # sample two dist randomly, set near and far wo biasing
        side = random.choice([-1, 1]) 
        dist_near = random.randint(int(screen_width/30), int(screen_width/4))
        dist_far = random.randint(int(screen_width/3.5), int(screen_width/2))
        
        cand_a_x = entry + (side * dist_near)
        cand_b_x = entry - (side * dist_far)
        
        # checking, just for fun
        if not (0 < cand_a_x < screen_width and 0 < cand_b_x < screen_width):
            continue

        # third level
        h3_goal_x = cand_b_x + random.randint(-50, 50)
        if not (0 < h3_goal_x < screen_width):
            continue

        # costs
        cost_g_a = greedy_func(entry, cand_a_x, h3_goal_x, c)
        cost_p_a = planning_func(entry, cand_a_x, h3_goal_x, c)
        
        cost_g_b = greedy_func(entry, cand_b_x, h3_goal_x, c)
        cost_p_b = planning_func(entry, cand_b_x, h3_goal_x, c)

        greedy_prefers_a = (cost_g_a * degree_of_conflict < cost_g_b)
        planner_prefers_b = (cost_p_b * degree_of_conflict < cost_p_a)
        
        if greedy_prefers_a and planner_prefers_b:
            trial = {
                "trial_id": len(experiment_configs) + 1,
                "levels": [
                    {"level": 1, "holes": [entry]},
                    {"level": 2, "holes": [cand_a_x, cand_b_x]},
                    {"level": 3, "holes": [h3_goal_x]}
                ],
                "metadata": {
                    "greedy_choice": cand_a_x,
                    "planner_choice": cand_b_x,
                    "switch_cost_param": c,
                    "cost_greedy_diff": cost_g_b - cost_g_a,
                    "cost_planner_diff": cost_p_a - cost_p_b
                }
            }
            experiment_configs.append(trial)
            
    return experiment_configs

In [4]:
levels = generate_levels(num_levels = 30)

with open('greedy_vs_2step.json', 'w') as f:
    json.dump(levels, f, indent=4)